# Playstyle analysis (EPL 2003/04 vs 2015/16)


Compare how Premier League playstyle changed between **2003/04** and **2015/16**.

**Headline questions:**
1. How many **passes per match**?
2. How **long are possessions** (events per possession, passes per possession, duration in seconds)?

Additional themes: pressing, direct play, physicality, chance quality.

**Note:** 2003/04 has only **38 matches** vs **380** in 2015/16 — use boxplots and treat means as illustrative.

## Per season analysis


In [13]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
import dill
import duckdb

warnings.filterwarnings("ignore")

# Repo root must be on sys.path before importing src (cwd varies by notebook folder).
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "config.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, RAW_DIR, NOTEBOOK_DIR, SRC_DIR

from src.analysis.metrics import (
    ERA_2004,
    ERA_2016,
    EVENT_COLS,
    build_match_team,
    build_season_metrics
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# dill.load_session('../saves/notebook_env.db')

In [14]:
matches_2004 = pd.read_csv(PROCESSED_DIR / "matches_2004.csv").assign(season=ERA_2004)
matches_2016 = pd.read_csv(PROCESSED_DIR / "matches_2016.csv").assign(season=ERA_2016)
matches_df = pd.concat([matches_2004, matches_2016], ignore_index=True)

events_2004_path = PROCESSED_DIR / "events_2004.parquet"
events_2016_path = PROCESSED_DIR / "events_2016.parquet"
events_df = pd.concat(
    [
        pd.read_parquet(events_2004_path, columns=EVENT_COLS),
        pd.read_parquet(events_2016_path, columns=EVENT_COLS),
    ],
    ignore_index=True,
)

In [15]:
match_meta = matches_df[["match_id", "season", "match_kickoff"]].drop_duplicates()
analysis_events = events_df.merge(match_meta, on="match_id", how="inner")

matches_by_season = match_meta.groupby("season").agg(matches=("match_id", "nunique"))
events_by_season = analysis_events.groupby("season").size().rename("events")

coverage = matches_by_season.join(events_by_season, how="left").fillna(0)
coverage["events"] = coverage["events"].astype(int)

print(f"Total matches in catalog: {match_meta['match_id'].nunique()}")
print(f"Total events captured: {len(analysis_events):,}\n")
print("Matches vs events captured by season:")
print(coverage)

if ERA_2016 in coverage.index:
    est_1516 = coverage.loc["2015/2016", "matches"] * 3404
    actual_1516 = coverage.loc["2015/2016", "events"]
    print(f"\n2015/16 expected ~{est_1516:,.0f} events | actual {actual_1516:,}")
if ERA_2004 in coverage.index:
    est_0304 = coverage.loc["2003/2004", "matches"] * 3404
    actual_0304 = coverage.loc["2003/2004", "events"]
    print(f"2003/04 expected ~{est_0304:,.0f} events | actual {actual_0304:,}")


Total matches in catalog: 418
Total events captured: 1,443,174

Matches vs events captured by season:
           matches   events
season                     
2003/2004       38   129401
2015/2016      380  1313773

2015/16 expected ~1,293,520 events | actual 1,313,773
2003/04 expected ~129,352 events | actual 129,401


In [16]:
match_team = build_match_team(analysis_events)
season_metrics = build_season_metrics(match_team)


# Extracting one match

Since the definitive teams during the 2003/2004 era were Arsene Wenger's Arsenal and Jose Mourinho's Chelsea, I plan to analyze either team's first match and compare their progression structure, playstyle, tempo etc. to their other matches over the season to see how it has progressed overall. I want to see how the team's tactical identity develops/shapes throughout the season. This would in turn tell us more about the playstyle that defined the "Barclay's" era.

I chose to analyze Arsenal's 2003/2004 season as they went unbeaten that season and went on to become champions. Arsenal's opening match of the season was vs. Everton with `match_id 3749493`. 

In [17]:
match_df = duckdb.sql("SELECT * FROM '../data/processed/events_2004.parquet' WHERE match_id = 3749493").df()
print(match_df.head())

  ball_receipt_outcome  ball_recovery_recovery_failure  block_deflection  \
0                 None                            <NA>              <NA>   
1                 None                            <NA>              <NA>   
2                 None                            <NA>              <NA>   
3                 None                            <NA>              <NA>   
4                 None                            <NA>              <NA>   

   block_offensive carry_end_location  clearance_aerial_won  \
0             <NA>               <NA>                  <NA>   
1             <NA>               <NA>                  <NA>   
2             <NA>               <NA>                  <NA>   
3             <NA>               <NA>                  <NA>   
4             <NA>               <NA>                  <NA>   

  clearance_body_part  clearance_head  clearance_left_foot  \
0                None            <NA>                 <NA>   
1                None            <NA>   

In [18]:
dill.dump_session('../saves/notebook_env.db')